# SQL Alerts no Databricks

## O que são SQL Alerts?

SQL Alerts são notificações automáticas disparadas quando o resultado de uma query atinge uma condição definida. Permitem monitorização proactiva de dados sem intervenção manual.

**Caso de uso típico:** Alertar quando vendas caem abaixo de um threshold, quando erros excedem um limite, ou quando dados não são actualizados.

---

## Componentes de um Alert

| Componente | Descrição |
|------------|-----------|
| **Query** | Consulta SQL que retorna o valor a monitorizar |
| **Trigger Condition** | Condição que dispara o alerta |
| **Refresh Schedule** | Frequência de execução da query |
| **Notification Destination** | Canal de notificação (email, Slack, webhook) |

---

## Trigger Conditions (Condições de Disparo)

A query deve retornar **uma única coluna e uma única linha** (valor escalar).

### Operadores Disponíveis

| Operador | Descrição | Exemplo |
|----------|-----------|---------|
| `>` | Maior que | Valor > 100 |
| `>=` | Maior ou igual | Valor >= 100 |
| `<` | Menor que | Valor < 50 |
| `<=` | Menor ou igual | Valor <= 50 |
| `=` | Igual a | Valor = 0 |
| `!=` | Diferente de | Valor != 0 |
| `is null` | Valor nulo | Valor is null |
| `is not null` | Valor não nulo | Valor is not null |

### Exemplo de Query para Alert

```sql
-- Conta registos com erro nas últimas 24h
SELECT COUNT(*) AS error_count
FROM logs
WHERE status = 'ERROR'
  AND timestamp > current_timestamp() - INTERVAL 24 HOURS
```

**Condição:** `error_count > 100` → dispara alerta

---

## Estados do Alert

| Estado | Ícone | Significado |
|--------|-------|-------------|
| **OK** | 🟢 | Condição não satisfeita |
| **TRIGGERED** | 🔴 | Condição satisfeita, alerta disparado |
| **UNKNOWN** | ⚪ | Query ainda não executou ou falhou |

### Fluxo de Estados

```
UNKNOWN → (primeira execução) → OK ou TRIGGERED
OK → (condição satisfeita) → TRIGGERED
TRIGGERED → (condição não satisfeita) → OK
```

**Para certificação:** O estado UNKNOWN indica que a query nunca executou com sucesso ou que houve erro na última execução.

---

## Notification Destinations

### Tipos Suportados

| Destino | Descrição |
|---------|-----------|
| **Email** | Envia para endereços configurados |
| **Slack** | Integração via webhook |
| **PagerDuty** | Para gestão de incidentes |
| **Microsoft Teams** | Via webhook connector |
| **Generic Webhook** | Qualquer endpoint HTTP |

### Configuração de Destinos

Os destinos são configurados a nível de **workspace**, não por alerta individual.

**Permissões necessárias:**
- Criar destinos: Workspace Admin
- Usar destinos existentes: Qualquer utilizador com permissão no alert

---

## Refresh Schedule (Agendamento)

### Opções de Frequência

| Intervalo | Caso de Uso |
|-----------|-------------|
| 1 minuto | Monitorização crítica em tempo real |
| 5 minutos | Métricas operacionais |
| 1 hora | KPIs de negócio |
| 1 dia | Relatórios diários |
| Personalizado | Expressão cron |

### Comportamento do Schedule

- A query executa no SQL Warehouse associado
- O warehouse **inicia automaticamente** se estiver parado
- Custo de DBUs incorrido a cada execução

**Para certificação:** Alerts com refresh frequente podem manter warehouses activos, aumentando custos.

---

## Criação de Alert — Passo a Passo

### 1. Criar a Query Base

```sql
-- Query deve retornar valor único
SELECT 
  CASE 
    WHEN COUNT(*) > 0 THEN COUNT(*)
    ELSE 0 
  END AS pending_orders
FROM orders
WHERE status = 'PENDING'
  AND created_at < current_timestamp() - INTERVAL 2 HOURS
```

### 2. Configurar o Alert

| Campo | Valor Exemplo |
|-------|---------------|
| Name | "Pedidos Pendentes > 2h" |
| Query | (selecionar query criada) |
| Trigger when | `pending_orders > 10` |
| Refresh | Every 15 minutes |
| Destination | email-operacoes |

### 3. Definir Template de Notificação (Opcional)

```
Alert: {{alert_name}}
Status: {{alert_status}}
Valor actual: {{query_result_value}}
Threshold: {{alert_threshold}}
Link: {{alert_url}}
```

---

## Mute (Silenciar) Alerts

### Quando Usar

- Manutenção planeada
- Janelas de deployment
- Investigação de falsos positivos

### Comportamento

| Acção | Alert Continua a Executar? | Notificações Enviadas? |
|-------|---------------------------|------------------------|
| Mute | ✅ Sim | ❌ Não |
| Unmute | ✅ Sim | ✅ Sim |
| Delete | ❌ Não | ❌ Não |

**Para certificação:** Mute suprime notificações mas **não para** a execução da query.

---

## Permissões e Ownership

### Níveis de Permissão

| Permissão | Pode Ver | Pode Editar | Pode Executar | Pode Transferir |
|-----------|----------|-------------|---------------|-----------------|
| **Viewer** | ✅ | ❌ | ❌ | ❌ |
| **Runner** | ✅ | ❌ | ✅ | ❌ |
| **Editor** | ✅ | ✅ | ✅ | ❌ |
| **Owner** | ✅ | ✅ | ✅ | ✅ |

### Transferência de Ownership

- Apenas o Owner actual ou Admin pode transferir
- Útil quando colaborador sai da equipa
- O novo owner precisa de acesso à query associada

## Referências Oficiais

- [Criar SQL Alerts](https://learn.microsoft.com/pt-pt/azure/databricks/sql/user/alerts/)
- [Notification Destinations](https://learn.microsoft.com/pt-pt/azure/databricks/sql/admin/notification-destinations)
- [Gerir Alerts](https://learn.microsoft.com/pt-pt/azure/databricks/sql/user/alerts/manage)